In [3]:
import gurobipy as gp
from gurobipy import GRB
import random
import numpy as np
import json
import csv
import time

# ── Parameters ────────────────────────────────────────────────────────────────
N_D = 10
N_C = 5
T_D = 20
T_C = 10
K = 10
SUB_GRID_SIZE = 3
NUM_OBSTACLES_SUBGRID = 16
MAX_EPOCHS = 900
THETA = 0.5
EPSILON = 0.01
GRID_SIZE = 10
FINE_GRID_SIZE = GRID_SIZE * SUB_GRID_SIZE
MAX_HEIGHT = 2
DELTA = 0.00001
MEASUREMENTS_PER_VISIT = 22

G = [(x, y) for x in range(GRID_SIZE) for y in range(GRID_SIZE)]

# ── Build Environment ─────────────────────────────────────────────────────────
def build_environment(seed=None):
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    mu_subgrid = {}
    obstacles_subgrid_all = {}
    interesting_coarse_cells = [(1,1),(2,2),(3,3),(4,4),(5,5),
                                 (6,6),(7,7),(8,8),(1,8),(8,1)]

    for coarse_x in range(GRID_SIZE):
        for coarse_y in range(GRID_SIZE):
            subgrid_cells = [(coarse_x * SUB_GRID_SIZE + si, coarse_y * SUB_GRID_SIZE + sj)
                             for si in range(SUB_GRID_SIZE) for sj in range(SUB_GRID_SIZE)]
            obstacles_subgrid_all[(coarse_x, coarse_y)] = []
            for cell in subgrid_cells:
                mu_subgrid[cell] = random.uniform(0.0, 0.2)

    for coarse_cell in interesting_coarse_cells:
        coarse_x, coarse_y = coarse_cell
        subgrid_cells = [(coarse_x * SUB_GRID_SIZE + si, coarse_y * SUB_GRID_SIZE + sj)
                         for si in range(SUB_GRID_SIZE) for sj in range(SUB_GRID_SIZE)]
        interesting_cell = random.choice(subgrid_cells)
        mu_subgrid[interesting_cell] = 0.95

    non_interesting = [c for c in G if c not in interesting_coarse_cells]
    obstacle_coarse_cells = random.sample(non_interesting, NUM_OBSTACLES_SUBGRID)
    for coarse_cell in obstacle_coarse_cells:
        coarse_x, coarse_y = coarse_cell
        subgrid_cells = [(coarse_x * SUB_GRID_SIZE + si, coarse_y * SUB_GRID_SIZE + sj)
                         for si in range(SUB_GRID_SIZE) for sj in range(SUB_GRID_SIZE)]
        valid_cells = [cell for cell in subgrid_cells if mu_subgrid[cell] != 0.95]
        if valid_cells:
            obstacle = random.choice(valid_cells)
            obstacles_subgrid_all[coarse_cell] = [obstacle]

    return mu_subgrid, obstacles_subgrid_all

# ── Low Level Planner - Fine ──────────────────────────────────────────────────
def low_level_planner_ip_low(X0_fine, Y0_fine, fine_epoch_goals, obstacles_subgrid):
    model = gp.Model("low_level_planner_ip_low")
    model.setParam('OutputFlag', 0)

    N_D_subgrid = len(X0_fine)
    N_C_subgrid = len(Y0_fine)
    num_cells = len(fine_epoch_goals)

    x = model.addVars(N_D_subgrid, T_D, K, num_cells, MAX_HEIGHT + 1, vtype=GRB.BINARY, name="x")
    y = model.addVars(N_C_subgrid, T_C, K, num_cells, vtype=GRB.BINARY, name="y")
    lam = model.addVars(K, vtype=GRB.BINARY, name="lam")
    visits = model.addVars(N_D_subgrid, num_cells, K, vtype=GRB.BINARY, name="visits")

    for i in range(N_D_subgrid):
        for l in range(num_cells):
            for k in range(K):
                model.addConstr(
                    visits[i, l, k] <= gp.quicksum(x[i, j, k, l, h]
                                                   for j in range(T_D)
                                                   for h in range(MAX_HEIGHT + 1)))
                for j in range(T_D):
                    for h in range(MAX_HEIGHT + 1):
                        model.addConstr(visits[i, l, k] >= x[i, j, k, l, h])
                model.addConstr(visits[i, l, k] <= lam[k])

    model.setObjective(gp.quicksum(lam[k] for k in range(K)), GRB.MINIMIZE)

    for k in range(K - 1):
        model.addConstr(lam[k] >= lam[k + 1])

    for i in range(N_D_subgrid):
        for k in range(K):
            model.addConstr(
                gp.quicksum(x[i, j, k, l, h]
                            for j in range(T_D)
                            for l in range(num_cells)
                            for h in range(MAX_HEIGHT + 1)) == T_D * lam[k])
            if X0_fine[i] in fine_epoch_goals:
                model.addConstr(
                    gp.quicksum(x[i, 0, k, fine_epoch_goals.index(X0_fine[i]), h]
                                for h in range(MAX_HEIGHT + 1)) >= lam[k])

    for i in range(N_D_subgrid):
        for k in range(K):
            model.addConstr(
                gp.quicksum(x[i, 0, k, l, h] for l in range(num_cells) for h in range(MAX_HEIGHT + 1))
                <= gp.quicksum(y[a, 0, k, l] for a in range(N_C_subgrid) for l in range(num_cells)))
            model.addConstr(
                gp.quicksum(x[i, T_D - 1, k, l, h] for l in range(num_cells) for h in range(MAX_HEIGHT + 1))
                <= gp.quicksum(y[a, T_C - 1, k, l] for a in range(N_C_subgrid) for l in range(num_cells)))

    for a in range(N_C_subgrid):
        for k in range(K):
            model.addConstr(
                gp.quicksum(y[a, b, k, l]
                            for b in range(T_C)
                            for l in range(num_cells)) == T_C * lam[k])
            if Y0_fine[a] in fine_epoch_goals:
                model.addConstr(y[a, 0, k, fine_epoch_goals.index(Y0_fine[a])] >= lam[k])

    for i in range(N_D_subgrid):
        for j in range(T_D - 1):
            for k in range(K):
                for l1 in range(num_cells):
                    x1, y1 = fine_epoch_goals[l1]
                    if (x1, y1) in obstacles_subgrid:
                        continue
                    valid_moves = [(x2, y2) for x2, y2 in fine_epoch_goals
                                   if abs(x2 - x1) + abs(y2 - y1) <= 1
                                   and (x2, y2) not in obstacles_subgrid]
                    for h1 in range(MAX_HEIGHT + 1):
                        valid_heights = [h2 for h2 in range(MAX_HEIGHT + 1) if abs(h2 - h1) <= 1]
                        model.addConstr(
                            x[i, j, k, l1, h1] <= gp.quicksum(
                                x[i, j + 1, k, fine_epoch_goals.index((x2, y2)), h2]
                                for x2, y2 in valid_moves for h2 in valid_heights))

    for a in range(N_C_subgrid):
        for b in range(T_C - 1):
            for k in range(K):
                for l1 in range(num_cells):
                    x1, y1 = fine_epoch_goals[l1]
                    valid_moves = [(x2, y2) for x2, y2 in fine_epoch_goals
                                   if abs(x2 - x1) + abs(y2 - y1) <= 1]
                    model.addConstr(
                        y[a, b, k, l1] <= gp.quicksum(
                            y[a, b + 1, k, fine_epoch_goals.index((x2, y2))]
                            for x2, y2 in valid_moves if (x2, y2) in fine_epoch_goals))

    for l, cell in enumerate(fine_epoch_goals):
        if cell not in obstacles_subgrid:
            model.addConstr(
                gp.quicksum(
                    x[i, j, k, l, h]
                    for i in range(N_D_subgrid)
                    for j in range(T_D)
                    for k in range(K)
                    for h in range(MAX_HEIGHT + 1)) >= 1)

    model.optimize()

    if model.status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL]:
        return None, {}, {}, [], {}, 0.0

    obj_val    = model.ObjVal
    lam_vals   = [lam[k].X for k in range(K)]
    X_vals     = {(i, j, k, l, h): x[i, j, k, l, h].X
                  for i in range(N_D_subgrid)
                  for j in range(T_D)
                  for k in range(K)
                  for l in range(num_cells)
                  for h in range(MAX_HEIGHT + 1)}
    Y_vals     = {(a, b, k, l): y[a, b, k, l].X
                  for a in range(N_C_subgrid)
                  for b in range(T_C)
                  for k in range(K)
                  for l in range(num_cells)}

    beta = 1.0
    d = N_D_subgrid + N_C_subgrid
    total_movement_cost = 0.0
    total_sensing_cost = 0.0
    T_pi = 0

    for i in range(N_D_subgrid):
        prev_pos = X0_fine[i] if X0_fine[i] in fine_epoch_goals else None
        for j in range(T_D):
            for k in range(K):
                for l in range(num_cells):
                    for h in range(MAX_HEIGHT + 1):
                        if X_vals.get((i, j, k, l, h), 0) > 0.5:
                            curr_pos = fine_epoch_goals[l]
                            if prev_pos:
                                total_movement_cost += ((curr_pos[0]-prev_pos[0])**2+(curr_pos[1]-prev_pos[1])**2)**0.5
                            prev_pos = curr_pos
                            T_pi += 1
                            total_sensing_cost += beta

    for a in range(N_C_subgrid):
        prev_pos = Y0_fine[a] if Y0_fine[a] in fine_epoch_goals else None
        for b in range(T_C):
            for k in range(K):
                for l in range(num_cells):
                    if Y_vals.get((a, b, k, l), 0) > 0.5:
                        curr_pos = fine_epoch_goals[l]
                        if prev_pos:
                            total_movement_cost += ((curr_pos[0]-prev_pos[0])**2+(curr_pos[1]-prev_pos[1])**2)**0.5
                        prev_pos = curr_pos
                        T_pi += 1

    total_economic_cost = total_movement_cost + (beta * d * T_pi) + total_sensing_cost

    visit_counts = {cell: 0 for cell in fine_epoch_goals}
    for cell in fine_epoch_goals:
        l = fine_epoch_goals.index(cell)
        for i in range(N_D_subgrid):
            for k in range(K):
                if visits[i, l, k].X > 0.5:
                    visit_counts[cell] += 1

    return obj_val, X_vals, Y_vals, lam_vals, visit_counts, total_economic_cost

# ── Low Level Planner - Coarse ────────────────────────────────────────────────
def low_level_planner_ip_high(X0, Y0, G, epoch_goals):
    model = gp.Model("low_level_planner_ip_high")
    model.setParam('OutputFlag', 0)

    N_D_actual = len(X0)
    N_C_actual = len(Y0)
    num_cells  = len(G)

    x = model.addVars(N_D_actual, T_D, K, num_cells, vtype=GRB.BINARY, name="x")
    y = model.addVars(N_C_actual, T_C, K, num_cells, vtype=GRB.BINARY, name="y")
    lam    = model.addVars(K, vtype=GRB.BINARY, name="lam")
    visits = model.addVars(N_D_actual, num_cells, K, vtype=GRB.BINARY, name="visits")

    for i in range(N_D_actual):
        for l in range(num_cells):
            for k in range(K):
                model.addConstr(
                    visits[i, l, k] <= gp.quicksum(x[i, j, k, l] for j in range(T_D)))
                for j in range(T_D):
                    model.addConstr(visits[i, l, k] >= x[i, j, k, l])
                model.addConstr(visits[i, l, k] <= lam[k])

    model.setObjective(gp.quicksum(lam[k] for k in range(K)), GRB.MINIMIZE)

    for k in range(K - 1):
        model.addConstr(lam[k] >= lam[k + 1])

    for i in range(N_D_actual):
        for k in range(K):
            model.addConstr(
                gp.quicksum(x[i, j, k, l] for j in range(T_D) for l in range(num_cells))
                == T_D * lam[k])
            start_l = G.index(X0[i])
            model.addConstr(x[i, 0, k, start_l] >= lam[k])

    for i in range(N_D_actual):
        for k in range(K):
            model.addConstr(
                gp.quicksum(x[i, 0, k, l] for l in range(num_cells))
                <= gp.quicksum(y[a, 0, k, l] for a in range(N_C_actual) for l in range(num_cells)))
            model.addConstr(
                gp.quicksum(x[i, T_D - 1, k, l] for l in range(num_cells))
                <= gp.quicksum(y[a, T_C - 1, k, l] for a in range(N_C_actual) for l in range(num_cells)))

    for a in range(N_C_actual):
        for k in range(K):
            model.addConstr(
                gp.quicksum(y[a, b, k, l] for b in range(T_C) for l in range(num_cells))
                == T_C * lam[k])
            start_l = G.index(Y0[a])
            model.addConstr(y[a, 0, k, start_l] >= lam[k])

    for i in range(N_D_actual):
        for j in range(T_D - 1):
            for k in range(K):
                for l1 in range(num_cells):
                    x1, y1 = G[l1]
                    valid_moves = [G.index((x2, y2)) for x2, y2 in G
                                   if abs(x2 - x1) + abs(y2 - y1) <= 1]
                    model.addConstr(
                        x[i, j, k, l1] <= gp.quicksum(x[i, j + 1, k, l2] for l2 in valid_moves))

    for a in range(N_C_actual):
        for b in range(T_C - 1):
            for k in range(K):
                for l1 in range(num_cells):
                    x1, y1 = G[l1]
                    valid_moves = [G.index((x2, y2)) for x2, y2 in G
                                   if abs(x2 - x1) + abs(y2 - y1) <= 1]
                    model.addConstr(
                        y[a, b, k, l1] <= gp.quicksum(y[a, b + 1, k, l2] for l2 in valid_moves))

    for goal in epoch_goals:
        lg = G.index(goal)
        model.addConstr(
            gp.quicksum(x[i, j, k, lg]
                        for i in range(N_D_actual)
                        for j in range(T_D)
                        for k in range(K)) >= 1)

    model.optimize()

    if model.status not in [GRB.OPTIMAL, GRB.SUBOPTIMAL]:
        return None, {}, {}, [], {}, 0.0

    obj_val    = model.ObjVal
    lam_vals   = [lam[k].X for k in range(K)]
    X_vals     = {(i, j, k, l): x[i, j, k, l].X
                  for i in range(N_D_actual)
                  for j in range(T_D)
                  for k in range(K)
                  for l in range(num_cells)}
    Y_vals     = {(a, b, k, l): y[a, b, k, l].X
                  for a in range(N_C_actual)
                  for b in range(T_C)
                  for k in range(K)
                  for l in range(num_cells)}

    beta = 1.0
    d = N_D_actual + N_C_actual
    total_movement_cost = 0.0
    total_sensing_cost = 0.0
    T_pi = 0

    for i in range(N_D_actual):
        prev_pos = X0[i] if X0[i] in G else None
        for j in range(T_D):
            for k in range(K):
                for l in range(num_cells):
                    if X_vals.get((i, j, k, l), 0) > 0.5:
                        curr_pos = G[l]
                        if prev_pos is not None:
                            dx = curr_pos[0] - prev_pos[0]
                            dy = curr_pos[1] - prev_pos[1]
                            total_movement_cost += (dx*dx + dy*dy)**0.5
                        prev_pos = curr_pos
                        T_pi += 1
                        if curr_pos in epoch_goals:
                            total_sensing_cost += beta

    for a in range(N_C_actual):
        prev_pos = Y0[a] if Y0[a] in G else None
        for b in range(T_C):
            for k in range(K):
                for l in range(num_cells):
                    if Y_vals.get((a, b, k, l), 0) > 0.5:
                        curr_pos = G[l]
                        if prev_pos is not None:
                            dx = curr_pos[0] - prev_pos[0]
                            dy = curr_pos[1] - prev_pos[1]
                            total_movement_cost += (dx*dx + dy*dy)**0.5
                        prev_pos = curr_pos
                        T_pi += 1

    total_economic_cost = total_movement_cost + (beta * d * T_pi) + total_sensing_cost

    visit_counts = {cell: 0 for cell in epoch_goals}
    for cell in epoch_goals:
        if cell in G:
            l = G.index(cell)
            for i in range(N_D_actual):
                for k in range(K):
                    if visits[i, l, k].X > 0.5:
                        visit_counts[cell] += 1

    return obj_val, X_vals, Y_vals, lam_vals, visit_counts, total_economic_cost

# ── Compute Coarse Stats ───────────────────────────────────────────────────────
def compute_coarse_stats(cell, measurements_fine):
    i, j = cell
    fine_cells = [(i * SUB_GRID_SIZE + si, j * SUB_GRID_SIZE + sj)
                  for si in range(SUB_GRID_SIZE) for sj in range(SUB_GRID_SIZE)]
    fine_mu_list, fine_U_list = [], []
    for f in fine_cells:
        n = len(measurements_fine[f])
        if n == 0:
            continue
        mu_hat_f = sum(measurements_fine[f]) / n
        U_f = 2 * np.sqrt((2 * np.log(np.log2(2 * n)) + np.log(12 * 9 / DELTA)) / (2 * n))
        fine_mu_list.append(mu_hat_f)
        fine_U_list.append(U_f)
    if not fine_mu_list:
        return 0.0, float('inf'), float('inf')
    k = len(fine_mu_list)
    mu_c = sum(fine_mu_list) / k
    U_c  = sum(fine_U_list)  / k
    return mu_c, U_c, mu_c + U_c

# ── High Level Planner - Fine ─────────────────────────────────────────────────
def high_level_planner_fine(coarse_cell, measurements_fine, keep_set, reject_set,
                             mu_subgrid, obstacles_subgrid_all):
    i, j = coarse_cell
    G_subgrid = [(i * SUB_GRID_SIZE + si, j * SUB_GRID_SIZE + sj)
                 for si in range(SUB_GRID_SIZE) for sj in range(SUB_GRID_SIZE)]
    obstacles_subgrid = obstacles_subgrid_all[coarse_cell]
    valid_subgrid = [cell for cell in G_subgrid if cell not in obstacles_subgrid]

    N_D_subgrid = min(N_D, len(valid_subgrid))
    if N_D_subgrid < 1:
        return False

    j_scores_fine = {}
    for cell in valid_subgrid:
        cell_measurements = measurements_fine[cell]
        n = len(cell_measurements)
        mu_hat = sum(cell_measurements) / n if n > 0 else 0.0
        U = float('inf') if n == 0 else 2 * np.sqrt(
            (2 * np.log(np.log2(2 * n)) + np.log(12 * 9 / DELTA)) / (2 * n))
        j_scores_fine[cell] = mu_hat + U

    ranked_fine_cells = sorted(j_scores_fine.items(), key=lambda x: x[1], reverse=True)
    fine_epoch_goals  = [cell for cell, _ in ranked_fine_cells[:N_D_subgrid]]
    X0_fine = fine_epoch_goals if len(fine_epoch_goals) <= N_D_subgrid else random.sample(fine_epoch_goals, N_D_subgrid)
    Y0_fine = X0_fine[:min(N_C, N_D_subgrid)]

    obj_val, X, Y, lam, visits, _ = low_level_planner_ip_low(
        X0_fine, Y0_fine, fine_epoch_goals, obstacles_subgrid)
    if obj_val is None:
        return False

    new_measurements = []
    for cell in fine_epoch_goals:
        l = fine_epoch_goals.index(cell)
        visited_pairs = set()
        for ii in range(N_D_subgrid):
            for k in range(K):
                hit = any(X.get((ii, t, k, l, h), 0) > 0.5
                          for t in range(T_D) for h in range(MAX_HEIGHT + 1))
                if hit:
                    visited_pairs.add((ii, k))
        for (ii, k) in visited_pairs:
            for _ in range(MEASUREMENTS_PER_VISIT):
                meas = np.random.binomial(1, mu_subgrid[cell])
                new_measurements.append((cell, meas))

    for cell, meas in new_measurements:
        measurements_fine[cell].append(meas)

    # KEEP: single fine cell — matches paper eq. 3
    for cell in fine_epoch_goals:
        n = len(measurements_fine[cell])
        if n == 0:
            continue
        mu_hat = sum(measurements_fine[cell]) / n
        U = 2 * np.sqrt((2 * np.log(np.log2(2 * n)) + np.log(12 * 9 / DELTA)) / (2 * n))
        if (mu_hat - U) >= (THETA - EPSILON):
            keep_set.add(coarse_cell)
            return True

    # REJECT: single fine cell early exit — same as original small
    for cell in fine_epoch_goals:
        n = len(measurements_fine[cell])
        if n == 0:
            continue
        mu_hat = sum(measurements_fine[cell]) / n
        U = 2 * np.sqrt((2 * np.log(np.log2(2 * n)) + np.log(12 * 9 / DELTA)) / (2 * n))
        if mu_hat + U <= THETA + EPSILON:
            reject_set.add(coarse_cell)
            return True

    return False

# ── High Level Planner - Coarse ───────────────────────────────────────────────
def high_level_planner_coarse(mu_subgrid, obstacles_subgrid_all):
    keep_set   = set()
    reject_set = set()
    measurements_fine = {cell: [] for cell in mu_subgrid.keys()}
    unlabeled_cells   = set(G)
    total_cost = 0.0
    last_epoch = 0

    for epoch in range(MAX_EPOCHS):
        last_epoch = epoch
        print(f"\n===== Coarse Epoch {epoch}: {len(unlabeled_cells)} unlabeled cells =====")

        j_scores = {}
        for cell in unlabeled_cells:
            mu_hat_c, U_c, J_c = compute_coarse_stats(cell, measurements_fine)
            j_scores[cell] = J_c

        ranked_cells = sorted(j_scores.items(), key=lambda x: x[1], reverse=True)
        top_cells    = [cell for cell, _ in ranked_cells[:min(N_D, len(ranked_cells))]]
        epoch_goals  = top_cells or []

        if len(top_cells) >= N_D:
            X0 = top_cells[:N_D]
        else:
            X0 = top_cells + ([top_cells[0]] * (N_D - len(top_cells)) if top_cells else [(0,0)] * N_D)
        Y0 = X0[:N_C]

        obj_val, X, Y, lam, visits, coarse_cost = low_level_planner_ip_high(X0, Y0, G, epoch_goals)
        if obj_val is None:
            continue

        total_cost += coarse_cost

        for cell in top_cells:
            if cell in unlabeled_cells and high_level_planner_fine(
                    cell, measurements_fine, keep_set, reject_set,
                    mu_subgrid, obstacles_subgrid_all):
                unlabeled_cells.discard(cell)

        # Secondary coarse check — aggregate over visited fine cells, matches paper eq. 4
        cells_to_remove = []
        for cell in unlabeled_cells:
            mu_hat_c, U_c, _ = compute_coarse_stats(cell, measurements_fine)
            if U_c == float('inf'):
                continue
            if (mu_hat_c - U_c) >= (THETA - EPSILON):
                keep_set.add(cell)
                cells_to_remove.append(cell)
            elif (mu_hat_c + U_c) <= (THETA + EPSILON):
                reject_set.add(cell)
                cells_to_remove.append(cell)
        for cell in cells_to_remove:
            unlabeled_cells.discard(cell)

        if len(unlabeled_cells) == 0:
            print("All cells labeled. Stopping.")
            break

    print(f"\nTotal Economic Cost: {total_cost:.4f}")
    print(f"Total Epochs Used: {last_epoch + 1}")
    print(f"KEEP SET ({len(keep_set)} cells): {sorted(list(keep_set))}")
    print(f"REJECT SET ({len(reject_set)} cells): {sorted(list(reject_set))}")

    return keep_set, reject_set, measurements_fine, last_epoch + 1, total_cost

# ── Accuracy ──────────────────────────────────────────────────────────────────
def compute_accuracy(keep_set, reject_set, mu_subgrid):
    correct = 0
    for cell in G:
        cx, cy = cell
        fine_cells = [(cx * SUB_GRID_SIZE + si, cy * SUB_GRID_SIZE + sj)
                      for si in range(SUB_GRID_SIZE) for sj in range(SUB_GRID_SIZE)]
        true_keep      = any(mu_subgrid[f] >= THETA for f in fine_cells)
        predicted_keep = cell in keep_set
        if true_keep == predicted_keep:
            correct += 1
    return correct / len(G)

# ── Stats Helper ──────────────────────────────────────────────────────────────
def stats(lst):
    a = np.array(lst, dtype=float)
    return {
        "mean"  : round(float(np.mean(a)),   4),
        "std"   : round(float(np.std(a)),    4),
        "min"   : round(float(np.min(a)),    4),
        "max"   : round(float(np.max(a)),    4),
        "median": round(float(np.median(a)), 4),
    }

# ── MAIN SIMULATION RUNNER ────────────────────────────────────────────────────
NUM_RUNS = 1
OUT_JSON = "all_runs_medium.json"
OUT_AVG  = "averaged_results_medium.json"
OUT_CSV  = "summary_medium.csv"

all_runs   = []
acc_list   = []
epoch_list = []
cost_list  = []
keep_list  = []
rej_list   = []

print(f"Starting {NUM_RUNS} simulation runs...\n")

for run_id in range(NUM_RUNS):
    seed = run_id * 7 + 42
    np.random.seed(seed)
    random.seed(seed)

    print(f"\n{'='*55}")
    print(f"  RUN {run_id + 1}/{NUM_RUNS}   (seed={seed})")
    print(f"{'='*55}")

    t0 = time.time()
    mu_subgrid, obstacles_subgrid_all = build_environment(seed=seed)

    keep_set, reject_set, measurements_fine, epochs_used, total_cost = \
        high_level_planner_coarse(mu_subgrid, obstacles_subgrid_all)

    elapsed  = time.time() - t0
    accuracy = compute_accuracy(keep_set, reject_set, mu_subgrid)

    run_record = {
        "run_id"      : run_id + 1,
        "seed"        : seed,
        "elapsed_sec" : round(elapsed, 2),
        "epochs_used" : epochs_used,
        "total_cost"  : round(total_cost, 4),
        "keep_count"  : len(keep_set),
        "reject_count": len(reject_set),
        "accuracy"    : round(accuracy, 4),
        "keep_set"    : sorted([list(c) for c in keep_set]),
        "reject_set"  : sorted([list(c) for c in reject_set]),
    }
    all_runs.append(run_record)
    acc_list.append(accuracy)
    epoch_list.append(epochs_used)
    cost_list.append(total_cost)
    keep_list.append(len(keep_set))
    rej_list.append(len(reject_set))

    print(f"\n  >> Run {run_id+1} Done:  Epochs={epochs_used}  Cost={total_cost:.2f}  Accuracy={accuracy:.4f}  Time={elapsed:.1f}s")

averaged = {
    "num_runs"    : NUM_RUNS,
    "epochs_used" : stats(epoch_list),
    "total_cost"  : stats(cost_list),
    "accuracy"    : stats(acc_list),
    "keep_count"  : stats(keep_list),
    "reject_count": stats(rej_list),
}

print("\n" + "="*55)
print("  AVERAGED RESULTS ACROSS ALL RUNS")
print("="*55)
for metric, val in averaged.items():
    if isinstance(val, dict):
        print(f"\n  {metric.upper()}:")
        for k, v in val.items():
            print(f"    {k:8s}: {v}")
    else:
        print(f"  {metric}: {val}")

with open(OUT_JSON, "w") as f:
    json.dump(all_runs, f, indent=2)
with open(OUT_AVG, "w") as f:
    json.dump(averaged, f, indent=2)
with open(OUT_CSV, "w", newline="") as f:
    fieldnames = ["run_id", "seed", "elapsed_sec", "epochs_used",
                  "total_cost", "keep_count", "reject_count", "accuracy"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in all_runs:
        writer.writerow({k: r[k] for k in fieldnames})

print(f"\nSaved: {OUT_JSON}, {OUT_AVG}, {OUT_CSV}")

Starting 1 simulation runs...


  RUN 1/1   (seed=42)

===== Coarse Epoch 0: 100 unlabeled cells =====

===== Coarse Epoch 1: 97 unlabeled cells =====

===== Coarse Epoch 2: 95 unlabeled cells =====

===== Coarse Epoch 3: 93 unlabeled cells =====

===== Coarse Epoch 4: 90 unlabeled cells =====

===== Coarse Epoch 5: 88 unlabeled cells =====

===== Coarse Epoch 6: 86 unlabeled cells =====

===== Coarse Epoch 7: 82 unlabeled cells =====

===== Coarse Epoch 8: 80 unlabeled cells =====

===== Coarse Epoch 9: 79 unlabeled cells =====

===== Coarse Epoch 10: 76 unlabeled cells =====

===== Coarse Epoch 11: 67 unlabeled cells =====

===== Coarse Epoch 12: 57 unlabeled cells =====

===== Coarse Epoch 13: 48 unlabeled cells =====

===== Coarse Epoch 14: 38 unlabeled cells =====

===== Coarse Epoch 15: 28 unlabeled cells =====

===== Coarse Epoch 16: 18 unlabeled cells =====

===== Coarse Epoch 17: 8 unlabeled cells =====
All cells labeled. Stopping.

Total Economic Cost: 70352.0000
Total Epochs